In [ ]:
# Worked Example: Date subtraction and float vs. Decimal CPI adjustment
import datetime
from decimal import Decimal

# 1. Parsing economic survey dates
base_date = datetime.date(2020, 1, 15)
current_date = datetime.date(2026, 6, 15)
days_elapsed = (current_date - base_date).days
print(f"Time elapsed between economic surveys: {days_elapsed} days")

# 2. Nominal wages and CPI figures
nominal_wage = 50_000.0
cpi_base = 100.0
cpi_current = 124.75

# Float calculation
real_wage_float = nominal_wage * (cpi_base / cpi_current)

# Exact Decimal calculation (pass numbers as strings to avoid float approximations)
wage_dec = Decimal("50000.00")
cpi_base_dec = Decimal("100.00")
cpi_curr_dec = Decimal("124.75")
real_wage_dec = wage_dec * (cpi_base_dec / cpi_curr_dec)

print(f"Float Real Wage:   ${real_wage_float:,.10f}")
print(f"Decimal Real Wage: ${real_wage_dec:,.10f}")
print(f"Discrepancy (cents): {(real_wage_float - float(real_wage_dec)) * 100:.4f} cents")

Time elapsed between economic surveys: 2343 days
Float Real Wage:   $40,080.1603206413
Decimal Real Wage: $40,080.1603206413
Discrepancy (cents): 0.0000 cents


In [ ]:
from decimal import Decimal
import datetime

def calculate_real_economic_value(
    nominal_value_str: str,
    base_cpi_str: str,
    current_cpi_str: str,
    start_date_str: str,
    end_date_str: str
) -> dict:
    """
    Calculates real economic purchasing power and inflation rate using Decimal arithmetic.
    """
    # Step 1: Parse date strings into datetime.date objects
    start_date = datetime.datetime.strptime(start_date_str, "%Y-%m-%d").date()
    end_date = datetime.datetime.strptime(end_date_str, "%Y-%m-%d").date()

    # Step 2: Validate that end_date is not earlier than start_date
    if end_date < start_date:
        raise ValueError("End date cannot precede start date.")

    # Step 3: Compute elapsed days between end_date and start_date
    elapsed_days = (end_date - start_date).days

    # Step 4: Convert numerical strings to Decimal objects
    nominal_val = Decimal(nominal_value_str)
    base_cpi = Decimal(base_cpi_str)
    current_cpi = Decimal(current_cpi_str)

    # Step 5: Calculate real value and CPI percentage change
    # Formula 1: Real Value = Nominal * (Base CPI / Current CPI)
    # Formula 2: CPI Change % = ((Current CPI - Base CPI) / Base CPI) * Decimal("100")
    real_val = nominal_val * (base_cpi / current_cpi)
    cpi_change_pct = ((current_cpi - base_cpi) / base_cpi) * Decimal("100")

    # Quantize to 2 decimal places for real_val and 4 for cpi_change_pct
    real_val = real_val.quantize(Decimal("0.00"))
    cpi_change_pct = cpi_change_pct.quantize(Decimal("0.0000"))

    # Step 6: Return the results in a dictionary
    return {
        "real_value": real_val,
        "cpi_change_pct": cpi_change_pct,
        "elapsed_days": elapsed_days
    }

In [ ]:
# Test cell for Exercise 1
# Run this cell to verify your implementation!

try:
    # Test case 1: Standard inflation adjustment (wages 60,000, CPI 100 -> 120)
    res1 = calculate_real_economic_value(
        nominal_value_str="60000.00",
        base_cpi_str="100.00",
        current_cpi_str="120.00",
        start_date_str="2020-01-01",
        end_date_str="2025-01-01"
    )
    assert res1["real_value"] == Decimal("50000.00"), f"Expected real_value Decimal('50000.00'), got {res1.get('real_value')}"
    assert res1["cpi_change_pct"] == Decimal("20.0000"), f"Expected cpi_change_pct Decimal('20.0000'), got {res1.get('cpi_change_pct')}"
    assert res1["elapsed_days"] == 1827, f"Expected 1827 elapsed days, got {res1.get('elapsed_days')}"

    # Test case 2: Deflationary adjustment (nominal 45,000, CPI 110 -> 99)
    res2 = calculate_real_economic_value(
        nominal_value_str="45000.00",
        base_cpi_str="110.00",
        current_cpi_str="99.00",
        start_date_str="2022-03-15",
        end_date_str="2024-03-15"
    )
    assert res2["real_value"] == Decimal("50000.00"), f"Expected real_value Decimal('50000.00'), got {res2.get('real_value')}"
    assert res2["cpi_change_pct"] == Decimal("-10.0000"), f"Expected cpi_change_pct Decimal('-10.0000'), got {res2.get('cpi_change_pct')}"

    # Test case 3: Invalid dates validation
    try:
        calculate_real_economic_value(
            nominal_value_str="50000.00",
            base_cpi_str="100.00",
            current_cpi_str="110.00",
            start_date_str="2025-01-01",
            end_date_str="2020-01-01"
        )
        assert False, "Expected ValueError for end_date preceding start_date."
    except ValueError:
        pass

    print("🎉 All Part 1 tests passed!")
except AssertionError as e:
    print(f"❌ Verification failed: {e}")
except Exception as e:
    print(f"❌ Unexpected error during testing: {type(e).__name__}: {e}")

🎉 All Part 1 tests passed!


In [ ]:
# Worked Example: Indicator set comparison and query routing
q1_indicators = {"GDP_US", "CPI_US", "UNEMP_US", "IND_PROD"}
q2_indicators = {"GDP_US", "CPI_US", "PAYROLLS_US", "RETAIL_SALES"}

added_series = q2_indicators.difference(q1_indicators)
retired_series = q1_indicators.difference(q2_indicators)
print(f"Newly added economic series: {added_series}")
print(f"Retired economic series:    {retired_series}")

# Simple pattern matching demonstration for macro queries
def route_macro_query(query: dict):
    match query:
        case {"indicator": ind, "frequency": "annual", "country": country}:
            return f"Fetching annual data for {ind} ({country})."
        case {"indicator": ind, "frequency": "monthly", "country": country}:
            return f"Fetching monthly data for {ind} ({country})."
        case _:
            return "Rejected: Unknown query format."

print(route_macro_query({"indicator": "GDP_US", "frequency": "annual", "country": "USA"}))

Newly added economic series: {'RETAIL_SALES', 'PAYROLLS_US'}
Retired economic series:    {'UNEMP_US', 'IND_PROD'}
Fetching annual data for GDP_US (USA).


In [ ]:
import random
import csv
from typing import Generator

def economic_tick_generator(start_price: float, volatility: float, steps: int) -> Generator[float, None, None]:
    """
    Simulates economic price tick stream via random walk.
    """
    random.seed(42)
    price = start_price

    # TODO: Loop 'steps' times, update price with normal shock, and yield price
    for _ in range(steps):
        # Draw normal shock: r = random.normalvariate(0, volatility)
        r = random.normalvariate(0, volatility)
        # Update price: price = price * (1.0 + r)
        price = price * (1.0 + r)
        # yield price
        yield price


def audit_macro_stream_and_log(price_feed, threshold: float, log_filename: str) -> dict:
    """
    Streams prices, logs threshold breaches into a CSV audit file, and returns summary stats in finally.
    """
    tick_count = 0
    max_price = None
    min_price = None

    try:
        with open(log_filename, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["Tick", "Price", "Status"])
            writer.writeheader()

            for price in price_feed:
                tick_count += 1

                # Step 1: Update min_price and max_price tracking
                # TODO: Check if min_price is None or price < min_price
                if min_price is None or price < min_price:
                    min_price = price
                # TODO: Check if max_price is None or price > max_price
                if max_price is None or price > max_price:
                    max_price = price

                # Step 2: Check if price exceeds threshold and log alert row
                # TODO: if price > threshold: writer.writerow({"Tick": tick_count, "Price": round(price, 4), "Status": "INFLATION_ALERT"})
                if price > threshold:
                    writer.writerow({"Tick": tick_count, "Price": round(price, 4), "Status": "INFLATION_ALERT"})

    finally:
        # Step 3: Return summary statistics dictionary in finally block
        return {
            "total_ticks": tick_count,
            "max_price": max_price,
            "min_price": min_price
        }

def process_and_route_economic_queries(queries: list[dict], allowed_indicators: set[str]) -> list[str]:
    """
    Validates and routes economic data queries via structural pattern matching (match/case).
    """
    routing_log = []

    for query in queries:
        # Step 1: Validate indicator is present in official allowed_indicators set
        indicator = query.get("indicator")
        if indicator not in allowed_indicators:
            routing_log.append("REJECTED: Indicator not in official registry.")
            continue

        # Step 2: Validate sample years is positive (> 0)
        years = query.get("years", 0)
        if years <= 0:
            routing_log.append("REJECTED: Sample years must be positive.")
            continue

        # Step 3: Match and route queries based on type and structure
        match query:
            case {"indicator": ind, "country": country, "years": yrs, "type": "annual"}:
                # TODO: Append the annual routing message
                routing_log.append(f"ROUTE_ANNUAL: Querying {ind} for {country} over {yrs} years.")

            case {"indicator": ind, "country": country, "years": yrs, "type": "quarterly", "seasonal_adj": adj}:
                # TODO: Check if adj is boolean using isinstance(adj, bool)
                # If not, append "REJECTED: Invalid query structure."
                # Otherwise, append f"ROUTE_QUARTERLY: Querying {ind} ({country}) over {yrs} years (SA={adj})."
                if not isinstance(adj, bool):
                    routing_log.append("REJECTED: Invalid query structure.")
                else:
                    routing_log.append(f"ROUTE_QUARTERLY: Querying {ind} ({country}) over {yrs} years (SA={adj}).")

            case {"indicator": ind, "country": country, "years": yrs, "type": "monthly", "base_year": base_yr}:
                # TODO: Check if base_yr < 1900
                # If base_yr < 1900, append "REJECTED: Invalid base year."
                # Otherwise, append f"ROUTE_MONTHLY: Querying {ind} ({country}) over {yrs} years (Base={base_yr})."
                if base_yr < 1900:
                    routing_log.append("REJECTED: Invalid base year.")
                else:
                    routing_log.append(f"ROUTE_MONTHLY: Querying {ind} ({country}) over {yrs} years (Base={base_yr}).")

            case _:
                # Fallback for unrecognized query structures
                routing_log.append("REJECTED: Invalid query structure.")

    return routing_log

In [ ]:
# Test cell for Exercise 2
# Run this cell to verify your implementation!

try:
    allowed_registry = {"GDP_US", "CPI_US", "UNEMP_EU", "RETAIL_UK", "PAYROLLS_US"}

    valid_queries = [
        {"indicator": "GDP_US", "country": "USA", "years": 10, "type": "annual"},
        {"indicator": "CPI_US", "country": "USA", "years": 5, "type": "quarterly", "seasonal_adj": True},
        {"indicator": "UNEMP_EU", "country": "EUR", "years": 3, "type": "monthly", "base_year": 2015}
    ]

    logs = process_and_route_economic_queries(valid_queries, allowed_registry)
    assert len(logs) == 3, f"Expected 3 routed logs, got {len(logs)}"
    assert logs[0] == "ROUTE_ANNUAL: Querying GDP_US for USA over 10 years.", f"Log 0 mismatch: {logs[0]}"
    assert logs[1] == "ROUTE_QUARTERLY: Querying CPI_US (USA) over 5 years (SA=True).", f"Log 1 mismatch: {logs[1]}"
    assert logs[2] == "ROUTE_MONTHLY: Querying UNEMP_EU (EUR) over 3 years (Base=2015).", f"Log 2 mismatch: {logs[2]}"

    # Edge case validations
    edge_queries = [
        {"indicator": "UNKNOWN_ID", "country": "USA", "years": 5, "type": "annual"}, # Ticker error
        {"indicator": "GDP_US", "country": "USA", "years": 0, "type": "annual"},      # Years <= 0
        {"indicator": "UNEMP_EU", "country": "EUR", "years": 3, "type": "monthly", "base_year": 1850}, # Base year < 1900
        {"indicator": "CPI_US", "country": "USA", "years": 5, "type": "quarterly", "seasonal_adj": "yes"}, # Non-bool SA
        {"indicator": "GDP_US", "country": "USA", "years": 5, "type": "weekly"}        # Invalid type
    ]

    edge_logs = process_and_route_economic_queries(edge_queries, allowed_registry)
    assert edge_logs[0] == "REJECTED: Indicator not in official registry.", f"Got {edge_logs[0]}"
    assert edge_logs[1] == "REJECTED: Sample years must be positive.", f"Got {edge_logs[1]}"
    assert edge_logs[2] == "REJECTED: Invalid base year.", f"Got {edge_logs[2]}"
    assert edge_logs[3] == "REJECTED: Invalid query structure.", f"Got {edge_logs[3]}"
    assert edge_logs[4] == "REJECTED: Invalid query structure.", f"Got {edge_logs[4]}"

    print("🎉 All Part 2 tests passed!")
except AssertionError as e:
    print(f"❌ Verification failed: {e}")
except Exception as e:
    print(f"❌ Unexpected error during testing: {type(e).__name__}: {e}")

🎉 All Part 2 tests passed!


In [ ]:
# Worked Example: Base class and inheritance for economic indicators

class EconomicIndicator:
    def __init__(self, code: str, category: str):
        self.code = code
        self.category = category

    def get_value(self, **kwargs) -> float:
        raise NotImplementedError("Subclasses must implement get_value()")

class SimpleMacroSeries(EconomicIndicator):
    def __init__(self, code: str, value: float):
        super().__init__(code, "Macroeconomic")
        self.value = value

    def get_value(self, **kwargs) -> float:
        return self.value

gdp = SimpleMacroSeries("GDP_US", 27.5) # $27.5 Trillion
print(f"{gdp.code} ({gdp.category}): {gdp.get_value()} Trillion USD")

GDP_US (Macroeconomic): 27.5 Trillion USD


In [ ]:
from typing import Mapping

# Step 1: Implement Custom Exception InvalidEconomicWeightsError
class InvalidEconomicWeightsError(Exception):
    """Exception raised when indicator weights do not sum to 1.0."""
    def __init__(self, current_sum: float, message: str = "Economic weights must sum to 1.0"):
        self.current_sum = current_sum
        super().__init__(f"{message} (Current sum: {current_sum:.4f})")


# Step 2: Implement MacroSeries subclassing EconomicIndicator
class MacroSeries(EconomicIndicator):
    def __init__(self, code: str, current_val: float, previous_val: float = 0.0):
        super().__init__(code, "Macroeconomic")
        self.current_val = current_val
        self.previous_val = previous_val

    @property
    def growth_rate(self) -> float:
        # Return (current_val - previous_val) / previous_val if previous_val > 0, else 0.0
        if self.previous_val > 0:
            return (self.current_val - self.previous_val) / self.previous_val
        return 0.0

    def get_value(self, **kwargs) -> float:
        # Return current_val
        return self.current_val


# Step 3: Implement SurveyDataset subclassing EconomicIndicator
class SurveyDataset(EconomicIndicator):
    def __init__(self, code: str, index_score: float, confidence_interval: float = 0.05):
        super().__init__(code, "Survey")
        self.index_score = index_score
        self.confidence_interval = confidence_interval

    def get_adjusted_score(self, risk_discount: float) -> float:
        # Formula: index_score * (1.0 - risk_discount * confidence_interval)
        # Return adjusted score
        return self.index_score * (1.0 - risk_discount * self.confidence_interval)

    def get_value(self, **kwargs) -> float:
        # Extract risk_discount from kwargs (default 0.1 if not passed)
        risk_discount = kwargs.get("risk_discount", 0.1)
        # Return self.get_adjusted_score(risk_discount)
        return self.get_adjusted_score(risk_discount)


# Step 4: Implement EconomicModel Aggregator
class EconomicModel:
    def __init__(self, indicator_weights: Mapping[EconomicIndicator, float]):
        # Compute current_sum of weights (sum(indicator_weights.values()))
        # If abs(current_sum - 1.0) >= 1e-6, raise InvalidEconomicWeightsError(current_sum)
        # Store self.indicator_weights = indicator_weights
        current_sum = sum(indicator_weights.values())
        if abs(current_sum - 1.0) >= 1e-6:
            raise InvalidEconomicWeightsError(current_sum)
        self.indicator_weights = indicator_weights

    def get_model_index(self, **kwargs) -> float:
        # Calculate weighted sum of indicator values
        # Hint: sum(weight * ind.get_value(**kwargs) for ind, weight in self.indicator_weights.items())
        return sum(weight * ind.get_value(**kwargs) for ind, weight in self.indicator_weights.items())

In [ ]:
# Test cell for Exercise 3
# Run this cell to verify your implementation!

try:
    # 1. Test MacroSeries
    gdp_series = MacroSeries("GDP_US", 105.0, 100.0)
    assert gdp_series.code == "GDP_US", "MacroSeries code attribute failed."
    assert gdp_series.category == "Macroeconomic", "MacroSeries category attribute failed."
    assert abs(gdp_series.growth_rate - 0.05) < 1e-6, f"Expected 0.05 growth rate, got {gdp_series.growth_rate}"
    assert gdp_series.get_value() == 105.0, f"Expected get_value() == 105.0, got {gdp_series.get_value()}"

    # 2. Test SurveyDataset
    sentiment = SurveyDataset("CONS_SENT", 80.0, 0.10)
    assert sentiment.code == "CONS_SENT", "SurveyDataset code attribute failed."
    assert sentiment.category == "Survey", "SurveyDataset category attribute failed."
    # Adjusted score with default risk_discount 0.1 -> 80.0 * (1 - 0.1 * 0.1) = 79.2
    assert abs(sentiment.get_value() - 79.2) < 1e-4, f"Expected adjusted value 79.2, got {sentiment.get_value(79.2)}"

    # 3. Test EconomicModel aggregation
    valid_weights = {gdp_series: 0.7, sentiment: 0.3}
    model = EconomicModel(valid_weights)
    idx_val = model.get_model_index()
    expected_idx = (0.7 * 105.0) + (0.3 * 79.2) # 73.5 + 23.76 = 97.26
    assert abs(idx_val - expected_idx) < 1e-4, f"Expected index {expected_idx}, got {idx_val}"

    # 4. Test Invalid Weights Exception
    invalid_weights = {gdp_series: 0.7, sentiment: 0.5} # Sums to 1.2
    try:
        EconomicModel(invalid_weights)
        assert False, "Expected InvalidEconomicWeightsError when weights sum to 1.2."
    except InvalidEconomicWeightsError as e:
        assert "1.2" in str(e) or "1.0" in str(e), f"Exception message should show current sum: {e}"

    print("🎉 All Part 3 tests passed!")
except AssertionError as e:
    print(f"❌ Verification failed: {e}")
except Exception as e:
    print(f"❌ Unexpected error during testing: {type(e).__name__}: {e}")

🎉 All Part 3 tests passed!


In [ ]:
# Worked Example: Generators and CSV logging
import csv

# Generator yielding price items
def simple_price_generator(n):
    base_price = 100.0
    for i in range(1, n + 1):
        yield round(base_price + i * 0.5, 2)

gen = simple_price_generator(3)
print(f"Next price from generator: {next(gen)}")
print(f"Next price from generator: {next(gen)}")

# Audit logging with context manager
with open("temp_macro_audit.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Tick", "Price", "Status"])
    writer.writerow([1, 105.50, "INFLATION_ALERT"])

print("Macro audit log created successfully.")

Next price from generator: 100.5
Next price from generator: 101.0
Macro audit log created successfully.


In [ ]:
import random
import csv
from typing import Generator

def economic_tick_generator(start_price: float, volatility: float, steps: int) -> Generator[float, None, None]:
    """
    Simulates economic price tick stream via random walk.
    """
    random.seed(42)
    price = start_price

    # TODO: Loop 'steps' times, update price with normal shock, and yield price
    for _ in range(steps):
        # Draw normal shock: r = random.normalvariate(0, volatility)
        r = random.normalvariate(0, volatility)
        # Update price: price = price * (1.0 + r)
        price = price * (1.0 + r)
        # yield price
        yield price


def audit_macro_stream_and_log(price_feed, threshold: float, log_filename: str) -> dict:
    """
    Streams prices, logs threshold breaches into a CSV audit file, and returns summary stats in finally.
    """
    tick_count = 0
    max_price = None
    min_price = None

    try:
        with open(log_filename, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["Tick", "Price", "Status"])
            writer.writeheader()

            for price in price_feed:
                tick_count += 1

                # Step 1: Update min_price and max_price tracking
                # TODO: Check if min_price is None or price < min_price
                if min_price is None or price < min_price:
                    min_price = price
                # TODO: Check if max_price is None or price > max_price
                if max_price is None or price > max_price:
                    max_price = price

                # Step 2: Check if price exceeds threshold and log alert row
                # TODO: if price > threshold: writer.writerow({"Tick": tick_count, "Price": round(price, 4), "Status": "INFLATION_ALERT"})
                if price > threshold:
                    writer.writerow({"Tick": tick_count, "Price": round(price, 4), "Status": "INFLATION_ALERT"})

    finally:
        # Step 3: Return summary statistics dictionary in finally block
        return {
            "total_ticks": tick_count,
            "max_price": max_price,
            "min_price": min_price
        }

In [ ]:
# Test cell for Exercise 4
# Run this cell to verify your implementation!

try:
    # 1. Verify generator output
    feed = economic_tick_generator(100.0, 0.01, 10)
    prices = list(feed)
    assert len(prices) == 10, f"Expected 10 prices, got {len(prices)}"
    assert abs(prices[0] - 100.245) < 0.01, f"Seed missing or initial price shock wrong: {prices[0]}"

    # 2. Run Audit processing on 1,000 tick stream
    large_feed = economic_tick_generator(100.0, 0.005, 1000)
    log_file = "macro_inflation_alerts.csv"
    stats = audit_macro_stream_and_log(large_feed, threshold=105.00, log_filename=log_file)

    assert stats["total_ticks"] == 1000, f"Expected 1000 ticks, got {stats['total_ticks']}"
    assert stats["max_price"] is not None and abs(stats["max_price"] - 118.8942) < 0.1, f"Expected max ~118.89, got {stats['max_price']}"
    assert stats["min_price"] is not None and abs(stats["min_price"] - 92.7901) < 0.1, f"Expected min ~92.79, got {stats['min_price']}"

    # 3. Verify CSV contents
    with open(log_file, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    assert len(rows) > 0, "Expected alert rows written to CSV."
    assert float(rows[0]["Price"]) > 105.0, f"Alert price should exceed 105.0, got {rows[0]['Price']}"
    assert rows[0]["Status"] == "INFLATION_ALERT", f"Status should be INFLATION_ALERT, got {rows[0]['Status']}"

    print("🎉 All Part 4 tests passed!")
except AssertionError as e:
    print(f"❌ Verification failed: {e}")
except Exception as e:
    print(f"❌ Unexpected error during testing: {type(e).__name__}: {e}")

🎉 All Part 4 tests passed!
